# Argentina (ARS) — Fixed Income Derivatives

Lecap (zero-coupon), Lecer (CER-linked), and Bonares (ARS sovereign).

**Key features:**
- Extreme rates: 40%+ policy rate
- CER: Coeficiente de Estabilización de Referencia (daily inflation)
- Lecap: capitalisation bonds (zero-coupon, short tenor)
- Lecer: CER-adjusted bonds (daily accrual)
- Bonares: ARS-denominated with semi-annual coupon

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "python"))

import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.fixed_income.argentine import (
    LecapBond, LecerBond, BONARBond,
    build_ars_curve, synthetic_ars_strip,
)
from pricebook.fixed_income.inflation_unit import InflationUnitBond, dual_curve_breakeven
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 11, 4)
print(f"Reference date: {REF}")
print(f"BCRA policy rate: 40%")
print(f"CER value: ~1,200")

## 1. ARS Discount Curve (Extreme Rates)

In [ ]:
strip = synthetic_ars_strip(REF, policy_rate=0.40)
ars_curve = build_ars_curve(REF, strip)

print(f"ARS Curve ({len(strip)} points) — extreme high rates:")
print(f"{'Tenor':>8}  {'Rate':>8}  {'DF':>10}")
print(f"{'─'*8}  {'─'*8}  {'─'*10}")
for c in strip:
    df = ars_curve.df(c["maturity"])
    print(f"{c['months']:>6}M  {c['rate']*100:>7.1f}%  {df:>10.6f}")

with apply_theme():
    fig, (ax1, ax2) = create_figure(2)
    tenors = [c["years"] for c in strip]
    ax1.plot(tenors, [c["rate"]*100 for c in strip], 'o-', linewidth=2, color='#dc2626')
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("Rate (%)")
    ax1.set_title("ARS Curve — Extreme Rates (40%+ policy)")

    ax2.plot(tenors, [ars_curve.df(c["maturity"]) for c in strip], 's-', linewidth=2, color='#dc2626')
    ax2.set_xlabel("Tenor (years)")
    ax2.set_ylabel("Discount Factor")
    ax2.set_title("ARS DFs (rapid decay at high rates)")
    fig.tight_layout()

## 2. Lecap — Zero-Coupon Capitalisation Bond

In [ ]:
print(f"Lecap Pricing (zero-coupon, various tenors at 40% rate):")
print(f"{'Tenor':>8}  {'Maturity':>12}  {'Price':>10}  {'(% of 1000)':>12}")
print(f"{'─'*8}  {'─'*12}  {'─'*10}  {'─'*12}")

for months in [1, 3, 6, 9, 12]:
    mat = REF + relativedelta(months=months)
    lecap = LecapBond(mat, 0.40)
    r = lecap.price(REF)
    print(f"{months:>6}M  {mat}  {r.price:>10.2f}  {r.price/10:>11.1f}%")

## 3. Lecer — CER-Linked Inflation Bond

In [ ]:
# Real ARS curve at ~5% (real terms)
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.interpolation import InterpolationMethod

real_dates = [REF + relativedelta(years=y) for y in [1, 2, 3, 5]]
real_dfs = [math.exp(-0.05 * y) for y in [1, 2, 3, 5]]
real_curve = DiscountCurve(REF, real_dates, real_dfs, interpolation=InterpolationMethod.LOG_LINEAR)

current_cer = 1200.0
cer_at_issue = 1100.0

print(f"Lecer Pricing (CER at issue={cer_at_issue}, current={current_cer}):")
print(f"  CER appreciation: {(current_cer/cer_at_issue - 1)*100:.1f}%\n")

for months in [3, 6, 9, 12]:
    issue = REF - relativedelta(months=3)
    mat = REF + relativedelta(months=months)
    lecer = LecerBond(issue, mat)
    r = lecer.price(REF, real_curve, current_cer, cer_at_issue)
    print(f"  {months}M to maturity: real={r.real_price:.2f}, nominal={r.nominal_price:.2f}, "
          f"ratio={r.nominal_price/r.real_price:.3f}")

## 4. Bonares — ARS Sovereign Bond

In [ ]:
print(f"BONAR Pricing (semi-annual, ACT/365, rates ~40%):")
print(f"{'Coupon':>8}  {'Tenor':>6}  {'Dirty Price':>12}  {'Note':>20}")
print(f"{'─'*8}  {'─'*6}  {'─'*12}  {'─'*20}")

for cpn in [0.10, 0.20, 0.30]:
    for t in [1, 2, 3, 5]:
        bonar = BONARBond(REF, REF + relativedelta(years=t), cpn)
        px = bonar.dirty_price(ars_curve)
        note = "deep discount" if px < 50 else "moderate" if px < 80 else "near par"
        print(f"{cpn*100:>7.0f}%  {t:>4}Y  {px:>11.4f}  {note:>20}")
    print()

## 5. Breakeven Inflation (ARS Nominal vs CER Real)

In [ ]:
bei = dual_curve_breakeven(ars_curve, real_curve, [1, 2, 3, 5])
print(f"Argentina Breakeven Inflation (extreme):")
print(f"{'Tenor':>6}  {'Nominal':>10}  {'Real':>8}  {'BEI':>10}")
print(f"{'─'*6}  {'─'*10}  {'─'*8}  {'─'*10}")
for row in bei:
    print(f"{row['years']:>4}Y  {row['nominal_rate']*100:>9.1f}%  "
          f"{row['real_rate']*100:>7.2f}%  {row['bei']*100:>9.1f}%")

with apply_theme():
    fig, axes = create_figure(1)
    ax = axes[0]
    years = [r["years"] for r in bei]
    ax.bar([y - 0.15 for y in years], [r["nominal_rate"]*100 for r in bei],
           width=0.3, label='Nominal (ARS)', color='#dc2626')
    ax.bar([y + 0.15 for y in years], [r["real_rate"]*100 for r in bei],
           width=0.3, label='Real (CER)', color='#059669')
    ax.set_xlabel("Tenor (years)")
    ax.set_ylabel("Rate (%)")
    ax.set_title("Argentina — Extreme BEI (~35% implied inflation)")
    ax.legend()
    fig.tight_layout()

## Summary

| Instrument | Convention | Key Feature |
|---|---|---|
| Lecap | Zero-coupon, ACT/365 | Short-term, 40%+ capitalisation |
| Lecer | CER-linked, daily accrual | Inflation-adjusted |
| Bonares | Semi-annual, ACT/365 | ARS sovereign (deep discount at 40%) |
| BEI | ARS - CER | ~35% implied inflation |